# Aria Analytics & Metrics Guide

**Complete guide for implementing analytics, tracking user behavior, business intelligence, and observability.**

Learn how to measure and optimize Aria's performance and user experience.


## Analytics Architecture

### Instrumentation Layers

```
Application
├─ User Actions (events)
├─ API Calls (metrics)
└─ Errors (exceptions)
    ↓
Collectors
├─ OpenTelemetry SDK
├─ Application Insights
└─ Custom tracers
    ↓
Processors
├─ Sampling
├─ Batching
└─ Enrichment
    ↓
Exporters
├─ Azure Monitor
├─ Datadog
└─ Custom backend
    ↓
Data Store
├─ Time-series DB
├─ Log Store
└─ Data Warehouse
    ↓
Visualization
├─ Dashboards
├─ Reports
└─ Alerting
```

---

## Event Tracking

### User Event Schema

```python
# shared/analytics.py
from dataclasses import dataclass
from datetime import datetime
import json

@dataclass
class AnalyticsEvent:
    event_type: str          # 'chat_message', 'character_action', 'model_inference'
    user_id: str
    session_id: str
    timestamp: datetime
    duration_ms: int         # How long the action took
    properties: dict         # Custom properties
    error: str = None        # If action failed

class EventTracker:
    def __init__(self, logger):
        self.logger = logger
        self.events = []

    def track_chat_message(
        self,
        user_id: str,
        session_id: str,
        model: str,
        prompt_tokens: int,
        completion_tokens: int,
        duration_ms: int,
        provider: str = "openai"
    ):
        """Track chat API call."""
        event = AnalyticsEvent(
            event_type="chat_message",
            user_id=user_id,
            session_id=session_id,
            timestamp=datetime.now(),
            duration_ms=duration_ms,
            properties={
                "model": model,
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "provider": provider,
                "total_tokens": prompt_tokens + completion_tokens,
                "cost_estimate": (prompt_tokens * 0.0005 + completion_tokens * 0.0015) / 1000  # GPT-4 pricing
            }
        )
        self.logger.info(json.dumps(event.__dict__, default=str))

    def track_model_inference(
        self,
        user_id: str,
        model: str,
        model_type: str,  # 'lora', 'base', 'quantum'
        duration_ms: int,
        tokens_generated: int,
        success: bool = True,
        error: str = None
    ):
        """Track model inference."""
        event = AnalyticsEvent(
            event_type="model_inference",
            user_id=user_id,
            session_id="",
            timestamp=datetime.now(),
            duration_ms=duration_ms,
            properties={
                "model": model,
                "model_type": model_type,
                "tokens_generated": tokens_generated,
                "success": success,
                "latency_ms": duration_ms
            },
            error=error
        )
        self.logger.info(json.dumps(event.__dict__, default=str))

    def track_character_action(
        self,
        user_id: str,
        action_type: str,  # 'move', 'gesture', 'pickup'
        target_entity: str,
        duration_ms: int
    ):
        """Track Aria character action."""
        event = AnalyticsEvent(
            event_type="character_action",
            user_id=user_id,
            session_id="",
            timestamp=datetime.now(),
            duration_ms=duration_ms,
            properties={
                "action_type": action_type,
                "target_entity": target_entity
            }
        )
        self.logger.info(json.dumps(event.__dict__, default=str))

# Global event tracker
event_tracker = EventTracker(logger)
```

### Integration in function_app.py

```python
# function_app.py
import time

@app.route("/api/chat/completions", methods=["POST"])
async def chat_completions(req: func.HttpRequest):
    """Chat completion with analytics."""
    start_time = time.time()
    user_id = req.headers.get("X-User-ID", "anonymous")
    session_id = req.headers.get("X-Session-ID", "")

    try:
        data = req.get_json()
        prompt = data.get("messages", "")

        # Call provider
        provider_name = detect_provider()
        provider = get_provider(provider_name)

        response = await provider.complete(prompt)

        # Track event
        duration_ms = (time.time() - start_time) * 1000
        event_tracker.track_chat_message(
            user_id=user_id,
            session_id=session_id,
            model=provider.model,
            prompt_tokens=estimate_tokens(prompt),
            completion_tokens=estimate_tokens(response),
            duration_ms=int(duration_ms),
            provider=provider_name
        )

        return func.HttpResponse(
            json.dumps({"response": response}),
            status_code=200
        )

    except Exception as e:
        duration_ms = (time.time() - start_time) * 1000
        event_tracker.track_chat_message(
            user_id=user_id,
            session_id=session_id,
            model="unknown",
            prompt_tokens=0,
            completion_tokens=0,
            duration_ms=int(duration_ms),
            provider="error"
        )
        raise
```

---

## Metrics & KPIs

### Key Performance Indicators

```python
# Metrics definitions
KPI_DEFINITIONS = {
    "chat_latency_p95": {
        "type": "histogram",
        "unit": "ms",
        "description": "95th percentile chat response latency",
        "threshold": 2000
    },
    "chat_cost_per_request": {
        "type": "gauge",
        "unit": "$",
        "description": "Average cost per chat request",
        "threshold": 0.01
    },
    "model_accuracy": {
        "type": "gauge",
        "unit": "%",
        "description": "Fine-tuned model accuracy on test set",
        "threshold": 0.85
    },
    "db_connection_pool_saturation": {
        "type": "gauge",
        "unit": "%",
        "description": "% of DB connections in use",
        "threshold": 0.80
    },
    "error_rate": {
        "type": "gauge",
        "unit": "%",
        "description": "% of requests that result in errors",
        "threshold": 0.01
    },
    "cache_hit_rate": {
        "type": "gauge",
        "unit": "%",
        "description": "% of cache hits vs total cache requests",
        "threshold": 0.80
    },
    "daily_active_users": {
        "type": "gauge",
        "unit": "count",
        "description": "Unique users per day",
        "threshold": 1000
    },
    "model_inference_latency": {
        "type": "histogram",
        "unit": "ms",
        "description": "LoRA model inference time",
        "threshold": 500
    }
}

# Telemetry collection
class MetricsCollector:
    def __init__(self, appinsights_key: str):
        self.client = TelemetryClient(appinsights_key)

    def track_metric(self, name: str, value: float, properties: dict = None):
        """Track a metric value."""
        self.client.track_metric(name, value, properties=properties)

    def track_histogram(self, name: str, value: float, count: int = 1):
        """Track histogram data."""
        self.client.track_metric(f"{name}_value", value)
        self.client.track_metric(f"{name}_count", count)

    def flush(self):
        """Send buffered metrics."""
        self.client.flush()
```

### Real-time Metrics Endpoint

```python
# function_app.py
@app.route("/api/metrics/realtime", methods=["GET"])
async def get_realtime_metrics(req: func.HttpRequest):
    """Get real-time system metrics."""
    return func.HttpResponse(
        json.dumps({
            "timestamp": datetime.now().isoformat(),
            "metrics": {
                "active_requests": len(active_requests),
                "db_connections_used": db_pool.checkedout(),
                "cache_hit_rate": cache.get_hit_rate(),
                "error_rate_5m": get_error_rate(window=300),
                "avg_latency_5m": get_avg_latency(window=300),
                "models_online": list_deployed_models(),
                "quantum_jobs_queued": quantum_queue_size()
            }
        }, default=str),
        status_code=200
    )
```

---

## Business Intelligence

### User Behavior Analysis

```python
# scripts/user_analytics.py
import pandas as pd
from sqlalchemy import create_engine

class UserAnalytics:
    def __init__(self, db_url: str):
        self.engine = create_engine(db_url)

    def get_user_engagement(self) -> pd.DataFrame:
        """Analyze user engagement metrics."""
        query = """
        SELECT
            u.id,
            u.username,
            COUNT(DISTINCT s.id) as sessions,
            COUNT(DISTINCT m.id) as messages,
            AVG(EXTRACT(EPOCH FROM (m.created_at - s.started_at))) as avg_session_duration,
            MAX(s.started_at) as last_active
        FROM users u
        LEFT JOIN sessions s ON u.id = s.user_id
        LEFT JOIN chat_messages m ON s.id = m.session_id
        WHERE u.created_at > NOW() - INTERVAL '30 days'
        GROUP BY u.id, u.username
        """
        return pd.read_sql(query, self.engine)

    def get_model_usage(self) -> pd.DataFrame:
        """Analyze which models are most used."""
        query = """
        SELECT
            model,
            COUNT(*) as usage_count,
            AVG(EXTRACT(EPOCH FROM created_at)) as avg_tokens_per_message
        FROM chat_messages
        WHERE created_at > NOW() - INTERVAL '7 days'
        GROUP BY model
        ORDER BY usage_count DESC
        """
        return pd.read_sql(query, self.engine)

    def get_revenue_metrics(self) -> dict:
        """Calculate revenue-related metrics."""
        query = """
        SELECT
            COUNT(DISTINCT user_id) as paying_users,
            SUM(CASE WHEN status = 'active' THEN 1 ELSE 0 END) as active_subscriptions,
            AVG(mrr) as avg_mrr
        FROM subscriptions
        WHERE created_at > NOW() - INTERVAL '30 days'
        """
        result = pd.read_sql(query, self.engine)
        return result.to_dict('records')[0]

    def export_dashboard_data(self, output_file: str):
        """Export data for BI dashboard."""
        engagement = self.get_user_engagement()
        model_usage = self.get_model_usage()
        revenue = self.get_revenue_metrics()

        with pd.ExcelWriter(output_file) as writer:
            engagement.to_excel(writer, sheet_name='Engagement')
            model_usage.to_excel(writer, sheet_name='Models')
            pd.DataFrame([revenue]).to_excel(writer, sheet_name='Revenue')
```

---

## Observability & Monitoring

### Distributed Tracing

```python
# shared/tracing.py
from opentelemetry import trace, metrics
from opentelemetry.exporter.azure_monitor import AzureMonitorTraceExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

# Setup tracer
exporter = AzureMonitorTraceExporter(
    connection_string=os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")
)
trace.set_tracer_provider(TracerProvider())
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(exporter)
)
tracer = trace.get_tracer(__name__)

# Usage in code
def process_chat_message(message: str):
    with tracer.start_as_current_span("chat_message_processing") as span:
        span.set_attribute("message.length", len(message))

        with tracer.start_as_current_span("tokenize") as child_span:
            tokens = tokenize(message)
            child_span.set_attribute("token.count", len(tokens))

        with tracer.start_as_current_span("model_inference") as child_span:
            response = infer(tokens)
            child_span.set_attribute("response.length", len(response))

        return response
```

### Log Aggregation

```python
# Structured logging for better analysis
import structlog

structlog.configure(
    processors=[
        structlog.stdlib.filter_by_level,
        structlog.stdlib.add_logger_name,
        structlog.stdlib.add_log_level,
        structlog.stdlib.PositionalArgumentsFormatter(),
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.StackInfoRenderer(),
        structlog.processors.format_exc_info,
        structlog.processors.UnicodeDecoder(),
        structlog.processors.JSONRenderer()
    ],
    context_class=dict,
    logger_factory=structlog.stdlib.LoggerFactory(),
    cache_logger_on_first_use=True,
)

logger = structlog.get_logger()

# Structured logging calls
logger.info(
    "chat_message_processed",
    user_id="user123",
    model="gpt-4",
    tokens_used=150,
    latency_ms=250,
    cost_usd=0.0045
)
```

---

## Alerting & Dashboards

### Alert Rules

```yaml
# config/alerts.yaml
alerts:
    - name: "High API Latency"
      metric: "chat_latency_p95"
      operator: ">"
      threshold: 2000
      duration: "5m"
      severity: "warning"
      action: "page_oncall"

    - name: "Database Connection Pool Saturation"
      metric: "db_connection_pool_saturation"
      operator: ">"
      threshold: 0.8
      duration: "2m"
      severity: "critical"
      action: "page_oncall"

    - name: "High Error Rate"
      metric: "error_rate"
      operator: ">"
      threshold: 0.05
      duration: "1m"
      severity: "critical"
      action: "page_oncall"

    - name: "Model Accuracy Degradation"
      metric: "model_accuracy"
      operator: "<"
      threshold: 0.85
      duration: "1h"
      severity: "warning"
      action: "email_team"
```

### Dashboard Queries (Kusto)

```kusto
// Chat API Performance
customMetrics
| where name == "chat_latency_p95"
| summarize avg(value), percentile(value, 95), max(value) by bin(timestamp, 1m)
| render timechart

// Error Rate Trend
customMetrics
| where name == "error_rate"
| summarize avg(value) by bin(timestamp, 5m)
| render barchart

// User Engagement
customEvents
| where name == "chat_message"
| summarize user_count = dcount(user_id), message_count = count() by tostring(bin(timestamp, 1d))
| render columnchart
```


## Analytics Best Practices Checklist

### Implementation

- [ ] Define KPIs before implementation
- [ ] Instrument at application entry points
- [ ] Use correlation IDs for distributed tracing
- [ ] Batch metrics to reduce overhead
- [ ] Implement sampling for high-volume events
- [ ] Monitor analytics infrastructure itself

### Data Quality

- [ ] Validate event schemas
- [ ] Remove PII before logging
- [ ] Use consistent timestamps (UTC)
- [ ] Document custom properties
- [ ] Regular data audits
- [ ] Maintain data retention policies

### Dashboarding

- [ ] Create role-specific views
- [ ] Include context in alerts
- [ ] Set realistic thresholds
- [ ] Regular dashboard reviews
- [ ] Mobile-friendly layouts
- [ ] Automated report generation

### Cost Management

- [ ] Use appropriate sampling rates
- [ ] Archive old data
- [ ] Optimize query performance
- [ ] Use cost allocation tags
- [ ] Regular cost reviews
- [ ] Optimize data volume
